# Quick start: Light in the Deep

This notebook is a first look at the toy telescope and its data. It simulates a small data set, saves it and reads it
back, and shows what is stored: the detector, the true event parameters, the pulses and hits of one event, the photon
bookkeeping, the hand-made features, and a file with several detectors.

Run it in the environment of the [README](../README.md) (`python -m pip install -e ".[analysis]"`), started with
`python -m jupyter lab` from the repository root, on the kernel `Python (fm-nu-telescope)`. It takes seconds on a CPU
and writes two small files to the repository's `data/` folder, which is `../data/` from this notebook's folder. The
physics is in [docs/physics.md](../docs/physics.md), the file layout in [docs/data_format.md](../docs/data_format.md),
and the tasks in [TASKS.md](../TASKS.md).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

import nutel
from nutel.toy import PRESETS, event_features, generate, image, load, preset, save
from nutel.toy.plot import draw_family, draw_geometry, event_display, photon_paths

print("nutel", nutel.__version__, "| torch", torch.__version__)

## Configuration

A configuration (`SimConfig`) has six parts: the layout of the detector, the medium, the readout, the physics switches,
the light emission and the prior of the events. The four presets form a ladder, each switching on more physics; the
medium (water or ice) is chosen separately, and every parameter can be changed on top.

In [ ]:
for name, cfg in PRESETS.items():
    ph, r = cfg.physics, cfg.effective_readout()  # the noise rate of "full" follows the medium
    print(
        f"{name:<9s} absorption={ph.absorption!s:<6s} scattering={ph.scattering!s:<6s} "
        f"stochastic={ph.stochastic!s:<6s} jitter={r.time_jitter:g} ns  noise={r.noise_rate:g} Hz"
    )

config = preset("full", medium="water")
print()
print(config.describe())

In [ ]:
custom = preset("scatter", medium="ice", noise_rate=20e3, n_strings=6)
print(custom.medium, "|", custom.layout.n_strings, "strings")

## The detector

A detector is a table of modules: string, module (0 = deepest) and position $(x, z)$ in metres, with $z$ up.

In [ ]:
geometry = config.geometry()
print(geometry.describe())

fig, ax = plt.subplots(figsize=(7, 4.5))
draw_geometry(ax, geometry)
ax.set_aspect("equal")
plt.show()

## Simulate

`generate` draws the events from the prior and tracks every photon until it is detected, absorbed or escapes. With
the default `device="auto"` it runs on a GPU if PyTorch sees one. `save_paths=3` keeps the photon paths of the first
three events.

In [ ]:
data = generate(2000, config=config, seed=1, save_paths=3)
data

In [ ]:
keys = ("seed", "device", "n_chunks", "package_version", "torch_version", "wall_time_s")
{key: data.provenance.get(key) for key in keys}

## What a data set holds

`truth` and `photons` have one entry per event; `pulses`, `hits` and `bursts` are ragged: the rows of event `i` are
`offset[i]:offset[i + 1]`. Every container returns its arrays by name with `arrays()`.

In [ ]:
for group in ("truth", "photons", "pulses", "hits"):
    print(group)
    for name, values in getattr(data, group).arrays().items():
        print(f"    {name:<22s} {values.dtype!s:<8s} {values.shape}")

The photon bookkeeping is exact: every emitted photon ends in exactly one way.

In [ ]:
p = data.photons
ended = p.n_detected + p.n_absorbed_module + p.n_absorbed + p.n_escaped + p.n_lost
print("every photon accounted for:", np.array_equal(ended, p.n_emitted))
print("detected fraction:", p.n_detected.sum() / p.n_emitted.sum())
print(
    "pulses per event by type (median):",
    {
        name: float(np.median(p.n_pulses[data.truth.type == k]))
        for k, name in enumerate(("track", "starting", "cascade", "atmospheric"))
    },
)

## Save and load

A file holds the data together with the detectors, the configuration and the provenance.

In [ ]:
path = save(data, "../data/quickstart_notebook.h5")
again = load(path)
print(again)
print("same configuration:", again.config == data.config)
print("same pulses:", np.array_equal(again.pulses.t, data.pulses.t))

## One event

`data.event(i)` gives the truth, the pulses, the hits and the bursts of one event. The display shows the hit modules
with an area proportional to their charge and a colour for their first time, the true track and the LineFit.

In [ ]:
i = int(np.flatnonzero(data.truth.type == 0)[0])  # the first through-going track
event = data.event(i)
print(event.type_name, {k: round(v, 3) if isinstance(v, float) else v for k, v in event.truth.items()})
pd.DataFrame({k: v for k, v in event.pulses.arrays().items() if k != "offset"}).head(10)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
event_display(ax, data, i, linefit=True)
plt.show()

The photon paths of one of the first events: direct light, scattered light and absorbed photons.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
photon_paths(ax, data, 0)
plt.show()

## Features and the module image

`event_features` turns every event into a row of hand-made features; `image` puts the pulses on the
`[string, module]` grid of the detector.

In [ ]:
features = pd.DataFrame(event_features(data))
features["type"] = data.truth.type_name
features.groupby("type")[["n_pulses", "total_charge", "duration", "x_rms", "z_rms"]].median()

In water every event carries about 24 noise hits spread uniformly over the 5 µs readout window, so the duration and
the extent of the hits are dominated by noise; task T3.3 removes them.

In [ ]:
img = image(data, channels=("t_first", "charge"))
print("image shape [events, channels, strings, modules]:", img.shape)

## Several detectors

With a detector family, `generate` samples several detectors and spreads the events over them. Every event points to
its detector with `truth.geometry_index`.

In [ ]:
multi = generate(
    600, config=preset("full", event_type="track"), seed=2, family="default", n_geometries=6, geometry_seed=3
)
print(multi)
for k, det in enumerate(multi.detectors):
    g, m = det.geometry, det.medium
    print(
        f"{k}: {g.layout:<9s} {g.n_strings:2d} strings, {g.n_modules:3d} modules, "
        f"lambda_a = {m.absorption_length:5.1f} m, lambda_eff = {m.scattering_length_eff:5.1f} m"
    )

In [ ]:
fig = draw_family(multi.detectors, ncols=3)
plt.show()

In [ ]:
table = pd.DataFrame(event_features(multi))
table.groupby("geometry_index")[["n_pulses", "total_charge", "n_modules"]].median()

## Next steps

- The tasks are in [TASKS.md](../TASKS.md): T0 to T4, and the bonus T5.
- `python examples/make_task_data.py` makes all their data sets; on a cluster, see
  [examples/cluster](../examples/cluster/README.md).